In [2]:
import re
import csv
import sys
from pathlib import Path
from typing import List, Dict

import pandas as pd

In [4]:
BLOCK_RE = re.compile(
    r"""===== \s*Running \s*on \s*(?P<path>[^\s=]+) \s*=====.*?      # path line
        Total \s+pairs \s+processed:\s*(?P<total_pairs>\d+).*?
        Successful \s+CLAP \s+calculations:\s*(?P<clap_success>\d+).*?
        Successful \s+PANNs \s+calculations:\s*(?P<panns_success>\d+).*?
        CLAP \s+Score \s+- \s+Mean:\s*(?P<clap_mean>[-\d.]+),\s*
             Min:\s*(?P<clap_min>[-\d.]+),\s*
             Max:\s*(?P<clap_max>[-\d.]+).*?
        PANNs \s+Symm \s+KL \s+- \s+Mean:\s*(?P<panns_mean>[-\d.]+),\s*
             Min:\s*(?P<panns_min>[-\d.]+),\s*
             Max:\s*(?P<panns_max>[-\d.]+).*?
        FAD \s+Score:\s*(?P<fad_score>[-\d.]+).*?
        KDA \s+Score:\s*(?P<kda_score>[-\d.]+)
    """,
    re.DOTALL | re.VERBOSE,
)


def extract_blocks(text: str):
    """Return a list of dictionaries for every summary block in text."""
    return [m.groupdict() for m in BLOCK_RE.finditer(text)]

def tf_analysis(log_path: Path, out_csv: Path | None):
    raw = log_path.read_text(encoding="utf-8", errors="ignore")
    records = extract_blocks(raw)

    if not records:
        print("No summary blocks found.")
        return

    df = pd.DataFrame(records)
    # convert numeric columns
    for col in df.columns:
        if col != "path":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if out_csv:
        df.to_csv(out_csv, index=False)
        print(f"Wrote {len(df)} rows → {out_csv}")
    else:
        print(df.to_string(index=False))


In [5]:
tf_analysis(Path('all_metrics_0727.log'), Path('all_metrics_0727.csv'))

Wrote 11 rows → all_metrics_0727.csv


In [16]:
df = pd.read_csv('all_metrics.csv')

In [17]:
df

,path,total_pairs,clap_success,panns_success,clap_mean,clap_min,clap_max,panns_mean,panns_min,panns_max,fad_score,kda_score
0,/tmp/tf_101,60,60,60,0.2148,-0.0545,0.4642,2.1714,0.1471,5.3931,11.1901,3.2159
1,/tmp/tf_109,60,60,60,0.2258,-0.0758,0.4466,2.3428,0.3807,5.5727,11.8841,3.6740
2,/tmp/tf_117,60,60,60,0.2415,-0.0559,0.5655,2.0501,0.1077,6.0160,12.8416,3.7878
3,/tmp/tf_125,60,60,60,0.2229,-0.1306,0.5032,2.2729,0.1636,5.2717,11.5416,4.1458
4,/tmp/tf_133,60,60,60,0.2143,-0.0944,0.5199,2.1319,0.1879,4.9218,12.1551,4.4085
5,/tmp/tf_141,60,60,60,0.2041,-0.2218,0.4498,2.1986,0.1707,6.4204,11.2441,3.9652
6,/tmp/tf_149,60,60,60,0.2162,-0.1343,0.4529,2.1433,0.2259,5.2651,12.9491,4.2512
7,/tmp/tf_157,60,60,60,0.2002,-0.0696,0.5273,2.0989,0.2002,5.6693,11.9541,3.6455
8,/tmp/tf_166,60,60,60,0.2244,-0.0184,0.5189,2.3766,0.0941,6.1978,12.5159,4.4282
9,/tmp/tf_174,60,60,60,0.1994,-0.1692,0.4953,2.2483,0.0804,5.6041,12.5181,4.3254
